# Point 2 — β Reliability & Geometry Robustness
Tests whether the encoding weight vectors (β) are stable across CV folds and
consistent between self/other conditions. Addresses the concern that geometric
conclusions derived from β vectors may be sensitive to feature specification.

In [ ]:
# ── CONFIG — edit here only ────────────────────────────────────────────────────
MODEL          = 'fasttext-wiki_worddur_xcirc'
LAYER          = 0
N_COMPONENTS   = 20

# Models to compare in Plot 4. Each entry is either a model tag (uses the
# global LAYER above) or a (model_tag, layer) tuple to override the layer —
# needed when comparing models with different layer counts/optima, e.g.
# bert-base-causal only has 13 layers (0-12) so it can't use LAYER=22.
# All five below use the same worddur+xcirc+--reliability convention, so
# they're directly comparable. fastText is a static (non-contextual) baseline —
# one fixed vector per word regardless of context — contrasted against the four
# contextual transformer models.
COMPARE_MODELS = [
    ('fasttext-wiki_worddur_xcirc', 0),               # fastText wiki-news-300d-1M, static baseline
    'bert-base-causal_ctx200spktag_worddur_xcirc',   # bert-base-causal, speaker-tagged (L12)
    'bert-base-causal_ctx200_worddur_xcirc',          # bert-base-causal, no speaker tag (L12)
    ('llama-3.1-8b_ctx200_worddur_xcirc', 22),
    ('gpt2-xl_ctx200_worddur_xcirc', 24),
]  # must have results

GLM_BASE  = '/scratch/aniluchavez/ConvoDATAS/SemanticGLM'
SCRIPT    = '/scratch/aniluchavez/hippocampal-speaker-semantics/scripts/semantic_glm.py'
CONDA_ENV = 'gpt2_embed'
PROJECT   = '/scratch/aniluchavez/hippocampal-speaker-semantics'

In [ ]:
import os, pickle, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats as spstats

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

def agg_path(model, pc, layer):
    return Path(GLM_BASE) / model / f'pc{pc}' / f'L{layer:02d}_all.pkl'

def patient_pkl_dir(model, pc):
    return Path(GLM_BASE) / model / f'pc{pc}'

def load_agg(model, pc=None, layer=None):
    pc = pc or N_COMPONENTS; layer = layer or LAYER
    p = agg_path(model, pc, layer)
    if not p.exists(): return None
    obj = pickle.load(open(p, 'rb'))
    return obj if isinstance(obj, pd.DataFrame) else obj.get('df', obj)

def load_patient_betas(model, pc=None, layer=None):
    """Load per-patient betas. Returns dict: patient_ID -> {(region,cond): (n_m, N_PC)}"""
    pc = pc or N_COMPONENTS; layer = layer or LAYER
    d = patient_pkl_dir(model, pc)
    out = {}
    for f in sorted(d.glob(f'*_L{layer:02d}_sem.pkl')):
        pid = f.name.split(f'_L{layer:02d}')[0]
        obj = pickle.load(open(f, 'rb'))
        if isinstance(obj, dict) and 'betas' in obj:
            out[pid] = obj['betas']  # {(region,cond): beta_mean}
    return out

In [ ]:
# Load aggregate results and betas for primary model
df_main = load_agg(MODEL)
betas   = load_patient_betas(MODEL)

if df_main is None:
    raise RuntimeError(f'No results for {MODEL} pc={N_COMPONENTS} layer={LAYER}. Run notebook 01 first.')

has_beta_corr = 'fold_beta_corr' in df_main.columns
has_betas     = len(betas) > 0
print(f'Loaded {len(df_main)} neuron-condition rows')
print(f'fold_beta_corr column: {has_beta_corr}')
print(f'Patient beta dicts: {len(betas)} patients')

In [ ]:
# ── PLOT 1: Cross-fold beta reliability distribution ──────────────────────────
if not has_beta_corr:
    print('fold_beta_corr not available — rerun semantic_glm.py with updated script')
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax_i, region in enumerate(['hippocampus', 'ACC']):
        ax = axes[ax_i]
        sub = df_main[df_main['region'] == region]
        for cond, color in [('self', '#2166ac'), ('other', '#4dac26')]:
            s = sub[sub['condition'] == cond]
            sig_corr  = s.loc[s['significant'],  'fold_beta_corr'].dropna()
            nsig_corr = s.loc[~s['significant'], 'fold_beta_corr'].dropna()
            x = np.linspace(-0.2, 1, 200)
            from scipy.stats import gaussian_kde
            if len(sig_corr) > 3:
                ax.plot(x, gaussian_kde(sig_corr)(x), color=color,
                        linewidth=2, label=f'{cond} (sig, n={len(sig_corr)})')
            if len(nsig_corr) > 3:
                ax.plot(x, gaussian_kde(nsig_corr)(x), color=color,
                        linewidth=1.5, linestyle='--', alpha=0.6,
                        label=f'{cond} (n.s., n={len(nsig_corr)})')
        ax.axvline(0, color='k', linewidth=0.8, linestyle=':')
        ax.set_xlabel('Cross-fold β correlation (mean pairwise Pearson r)')
        ax.set_ylabel('Density')
        ax.set_title(f'{region}  — β reliability')
        ax.legend(fontsize=8, frameon=False)
    plt.suptitle(f'{MODEL}  L{LAYER:02d}  pc={N_COMPONENTS}', y=1.01)
    plt.tight_layout()
    plt.savefig(f'../figures/02_fold_beta_corr_{MODEL}_L{LAYER:02d}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── PLOT 2: β reliability vs R² — do reliable neurons have better R²? ─────────
if has_beta_corr:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax_i, region in enumerate(['hippocampus', 'ACC']):
        ax = axes[ax_i]
        sub = df_main[df_main['region'] == region].dropna(subset=['fold_beta_corr','r2'])
        for cond, color in [('self', '#2166ac'), ('other', '#4dac26')]:
            s = sub[sub['condition'] == cond]
            sig  = s[s['significant']]
            nsig = s[~s['significant']]
            ax.scatter(nsig['fold_beta_corr'], nsig['r2'], color=color,
                       alpha=0.15, s=12, label=f'{cond} n.s.')
            ax.scatter(sig['fold_beta_corr'],  sig['r2'],  color=color,
                       alpha=0.8,  s=25, edgecolors='k', linewidths=0.4,
                       label=f'{cond} sig')
        ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
        ax.axvline(0, color='k', linewidth=0.8, linestyle=':')
        ax.set_xlabel('Cross-fold β correlation')
        ax.set_ylabel('Deviance R² (test)')
        ax.set_title(f'{region}')
        ax.legend(fontsize=8, frameon=False, markerscale=1.5)
    plt.suptitle(f'{MODEL}  L{LAYER:02d} — β reliability vs R²', y=1.01)
    plt.tight_layout()
    plt.savefig(f'../figures/02_beta_vs_r2_{MODEL}_L{LAYER:02d}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── PLOT 3: Self vs Other β alignment (Pearson correlation) ─────────────────────
# For each neuron significant in BOTH self and other, compare β directions
if not has_betas:
    print('No beta dicts found — run with updated script')
else:
    # fold_beta_corr per (patient, region, neuron_idx, condition) — needed below
    # to compare self/other similarity against each condition's own split-half
    # (cross-fold) reliability ceiling.
    rel = df_main.set_index(['patient', 'region', 'neuron_idx', 'condition'])['fold_beta_corr']

    cos_sims = {'hippocampus': [], 'ACC': []}
    ceiling_rows = []
    for pid, beta_dict in betas.items():
        for region in ['hippocampus', 'ACC']:
            if (region, 'self') not in beta_dict or (region, 'other') not in beta_dict:
                continue
            b_self  = beta_dict[(region, 'self')]   # (n_m, N_PC)
            b_other = beta_dict[(region, 'other')]  # (n_m, N_PC)
            if b_self.shape != b_other.shape: continue
            # Per-neuron Pearson correlation between the final (fold-averaged) self and other beta vectors
            cs = np.array([
                spstats.pearsonr(b_self[m], b_other[m])[0]
                if b_self[m].std() > 0 and b_other[m].std() > 0 else np.nan
                for m in range(b_self.shape[0])
            ])
            cos_sims[region].extend(cs.tolist())

            for m in range(b_self.shape[0]):
                if np.isnan(cs[m]):
                    continue
                try:
                    rel_self  = rel.loc[(pid, region, m, 'self')]
                    rel_other = rel.loc[(pid, region, m, 'other')]
                except KeyError:
                    continue
                ceiling_rows.append({
                    'patient': pid, 'region': region, 'neuron_idx': m,
                    'cos_sim': cs[m], 'rel_self': rel_self, 'rel_other': rel_other,
                })

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax_i, region in enumerate(['hippocampus', 'ACC']):
        ax = axes[ax_i]
        cs = np.array(cos_sims[region])
        cs = cs[~np.isnan(cs)]
        if len(cs) == 0:
            ax.set_visible(False); continue
        ax.hist(cs, bins=40, color='#2166ac', edgecolor='white', linewidth=0.3)
        ax.axvline(cs.mean(), color='k', linewidth=1.5, linestyle='--',
                   label=f'mean={cs.mean():.3f}')
        ax.axvline(0, color='grey', linewidth=0.8, linestyle=':')
        t, p = spstats.ttest_1samp(cs, 0)
        ax.set_title(f'{region}  — self/other β alignment\nt={t:.2f}  p={p:.3g}')
        ax.set_xlabel('Pearson correlation (self β vs other β)')
        ax.set_ylabel('Neuron count')
        ax.legend(fontsize=9, frameon=False)
    plt.suptitle(f'{MODEL}  L{LAYER:02d}  pc={N_COMPONENTS}', y=1.01)
    plt.tight_layout()
    plt.savefig(f'../figures/02_self_other_alignment_{MODEL}_L{LAYER:02d}.pdf', bbox_inches='tight')
    plt.show()

    # ── PLOT 3b: self/other similarity vs split-half reliability ceiling ──────
    # "Above chance" = cos_sim > 0 (tested above). "Below ceiling" = cos_sim
    # can't exceed what each condition's own beta reliability allows, since a
    # beta that doesn't even replicate across its own CV folds can't be
    # meaningfully aligned with the other condition's beta. Ceiling here is
    # the standard disattenuation bound sqrt(rel_self * rel_other); only
    # neurons with positive reliability in both conditions have a meaningful
    # (real-valued) ceiling.
    cdf = pd.DataFrame(ceiling_rows)
    cdf = cdf[(cdf['rel_self'] > 0) & (cdf['rel_other'] > 0)].copy()
    cdf['ceiling']       = np.sqrt(cdf['rel_self'] * cdf['rel_other'])
    cdf['disattenuated'] = cdf['cos_sim'] / cdf['ceiling']
    cdf['below_ceiling'] = cdf['cos_sim'] <= cdf['ceiling']

    print(f'Reliability-ceiling test ({len(cdf)}/{sum(len(v) for v in cos_sims.values())} '
          f'neurons have positive self & other reliability):')
    for region in ['hippocampus', 'ACC']:
        sub = cdf[cdf['region'] == region]
        if sub.empty:
            print(f'  {region}: no neurons with positive reliability in both conditions')
            continue
        t_above, p_above = spstats.ttest_1samp(sub['cos_sim'], 0)
        t_below, p_below = spstats.ttest_1samp(sub['cos_sim'] - sub['ceiling'], 0)
        p_below_one_sided = p_below / 2 if t_below < 0 else 1 - p_below / 2
        print(f'  {region}  n={len(sub)}\n'
              f'    above chance (cos_sim > 0):    t={t_above:.2f}  p={p_above:.3g}\n'
              f'    below ceiling (cos_sim < sqrt(rel_self*rel_other)): '
              f'{100*sub["below_ceiling"].mean():.1f}% of neurons, '
              f'mean gap={(sub["ceiling"]-sub["cos_sim"]).mean():.3f}, '
              f't={-t_below:.2f}  p(one-sided)={p_below_one_sided:.3g}\n'
              f'    median disattenuated r (cos_sim/ceiling)={sub["disattenuated"].median():.3f}')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax_i, region in enumerate(['hippocampus', 'ACC']):
        ax = axes[ax_i]
        sub = cdf[cdf['region'] == region]
        if sub.empty:
            ax.set_visible(False); continue
        ax.scatter(sub['ceiling'], sub['cos_sim'], s=14, alpha=0.4, color='#2166ac')
        lim = max(sub['ceiling'].max(), sub['cos_sim'].max(), 0.1)
        ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='cos_sim = ceiling')
        ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
        ax.set_xlabel('Reliability ceiling  √(rel_self × rel_other)')
        ax.set_ylabel('Self–other Pearson correlation')
        ax.set_title(f'{region}  ({100*sub["below_ceiling"].mean():.0f}% below ceiling)')
        ax.legend(fontsize=8, frameon=False)
    plt.suptitle(f'{MODEL}  L{LAYER:02d} — self/other similarity vs split-half ceiling', y=1.01)
    plt.tight_layout()
    plt.savefig(f'../figures/02_self_other_vs_ceiling_{MODEL}_L{LAYER:02d}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── PLOT 3c: Self-other beta correlation — ranked bar plot (REAL split-half reliability) ──
# Uses the actual reliability.py-derived r_cross/self_reliability_mean/other_reliability_mean
# (saved by semantic_glm.py --reliability), not the fold_beta_corr-derived proxy above.
def load_patient_reliability(model, pc=None, layer=None):
    """Load per-patient reliability dicts -> rows: patient, region, neuron_idx, r_cross, ..."""
    pc = pc or N_COMPONENTS; layer = layer or LAYER
    d = patient_pkl_dir(model, pc)
    rows = []
    for f in sorted(d.glob(f'*_L{layer:02d}_sem.pkl')):
        pid = f.name.split(f'_L{layer:02d}')[0]
        obj = pickle.load(open(f, 'rb'))
        if not (isinstance(obj, dict) and 'reliability' in obj):
            continue
        for region, neuron_list in obj['reliability'].items():
            for r in neuron_list:
                rows.append({
                    'patient': pid, 'region': region, 'neuron_idx': r['neuron'],
                    'r_cross': r['r_cross'],
                    'self_reliability_mean': r['self_reliability_mean'],
                    'other_reliability_mean': r['other_reliability_mean'],
                    'ceil_mean': r['ceil_mean'],
                    'null_distribution': np.asarray(r.get('null_distribution', [])),
                })
    return pd.DataFrame(rows)

rel_df = load_patient_reliability(MODEL)
has_reliability = not rel_df.empty

if not has_reliability:
    print('No reliability data found — rerun semantic_glm.py with --reliability')
else:
    import seaborn as sns
    valid = rel_df.dropna(subset=['r_cross', 'ceil_mean'])
    below = valid['r_cross'] <= valid['ceil_mean']
    print(f'below ceiling: {100*below.mean():.1f}% (n={len(valid)})')

    df_plot = rel_df.dropna(subset=['r_cross'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (region, color) in zip(axes, [('hippocampus', 'coral'), ('ACC', 'mediumseagreen')]):
        df_r = df_plot[df_plot['region'] == region].copy()
        if df_r.empty:
            ax.set_visible(False); continue
        df_r = df_r.sort_values('r_cross', ascending=False).reset_index(drop=True)
        df_r['rank'] = df_r.index + 1
        sns.barplot(data=df_r, x='rank', y='r_cross', color=color, ax=ax)
        ax.axhline(0, color='black', linewidth=1)
        ax.set_title(f'Self-Other Beta Correlation (split-half r_cross) — {region}\n{MODEL}  L{LAYER:02d}')
        ax.set_xlabel('Neuron rank')
        ax.set_ylabel('r_cross (self β vs other β, full-data fit)')
        ax.tick_params(axis='x', labelbottom=False)
    plt.tight_layout()
    plt.savefig(f'../figures/02_beta_corr_bar_reliability_{MODEL}_L{LAYER:02d}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── PLOT 3d: Speaking / Listening / Cross correlation violin (REAL split-half reliability) ──
if not has_reliability:
    print('No reliability data found — run the Plot 3c cell first (needs --reliability run)')
else:
    metric_labels = {
        'self_reliability_mean':  'Speaking',
        'other_reliability_mean': 'Listening',
        'r_cross':                'Cross correlation',
    }
    order   = ['Speaking', 'Listening', 'Cross correlation']
    palette = {'Speaking': '#4C78A8', 'Listening': '#F58518', 'Cross correlation': '#54A24B'}

    violin_df = rel_df.melt(
        id_vars=['patient', 'region', 'neuron_idx'],
        value_vars=['self_reliability_mean', 'other_reliability_mean', 'r_cross'],
        var_name='metric', value_name='value',
    )
    violin_df['metric'] = violin_df['metric'].map(metric_labels)
    violin_df = violin_df.dropna(subset=['value'])

    regions_present = [r for r in ['hippocampus', 'ACC'] if (rel_df['region'] == r).any()]
    fig, axes = plt.subplots(1, len(regions_present),
                              figsize=(6.5 * len(regions_present), 5.5), sharey=True)
    if len(regions_present) == 1:
        axes = [axes]
    for ax, region in zip(axes, regions_present):
        rdf = violin_df[violin_df['region'] == region]
        sns.violinplot(data=rdf, x='metric', y='value', order=order, hue='metric',
                        palette=palette, legend=False, inner='quartile', cut=0, linewidth=1, ax=ax)
        sns.stripplot(data=rdf, x='metric', y='value', order=order,
                       color='black', alpha=0.22, size=2.5, jitter=0.18, ax=ax)
        ax.axhline(0, color='black', linewidth=1)
        ax.set_title(f'{region} reliability')
        ax.set_xlabel('')
        ax.set_ylabel('Reliability / correlation')
        ax.tick_params(axis='x', rotation=20)
    fig.suptitle(f'Speaking, listening, and cross beta correlation (split-half) — {MODEL}  L{LAYER:02d}', y=1.03)
    plt.tight_layout()
    plt.savefig(f'../figures/02_three_violin_reliability_real_{MODEL}_L{LAYER:02d}.pdf', bbox_inches='tight')
    plt.show()

    # ── Significance tests: is Cross correlation above null, and below each ceiling? ──
    # "Above null": pool each neuron's own permutation null distribution for r_cross
    # (built by reliability.py via row-permuting the 'other' side and refitting) into
    # a null distribution of the REGION-MEAN r_cross, then locate the observed mean
    # in that null (one-sided, matches reliability.py's analyze_reliability_results).
    # "Below ceiling": paired one-sided t-test per neuron of (r_cross - Speaking) and
    # (r_cross - Listening) — i.e. is cross-condition correlation reliably smaller than
    # EACH condition's own within-condition split-half reliability individually.
    print()
    for region in regions_present:
        rdf_r = rel_df[rel_df['region'] == region].dropna(subset=['r_cross'])
        if rdf_r.empty:
            continue
        print(f'=== {region} (n={len(rdf_r)} neurons) ===')

        nulls = [nd for nd in rdf_r['null_distribution'] if nd is not None and len(nd) > 0]
        if nulls:
            min_n = min(len(nd) for nd in nulls)
            null_mat = np.vstack([nd[:min_n] for nd in nulls])
            R_null = null_mat.mean(axis=0)         # null distribution of region-mean r_cross
            R_obs  = rdf_r['r_cross'].mean()
            p_null = (np.sum(R_null >= R_obs) + 1) / (len(R_null) + 1)
            print(f'  above null:  R_obs={R_obs:.3f}  null={R_null.mean():.3f}±{R_null.std():.3f}  '
                  f'p(one-sided)={p_null:.3g}  (n_null={min_n})')
        else:
            print('  above null:  no null distributions available (rerun with --reliability)')

        for ceiling_col, label in [('self_reliability_mean', 'Speaking'),
                                    ('other_reliability_mean', 'Listening')]:
            sub = rdf_r.dropna(subset=[ceiling_col])
            diff = sub['r_cross'] - sub[ceiling_col]
            t, p_two = spstats.ttest_1samp(diff, 0)
            p_one = p_two / 2 if t < 0 else 1 - p_two / 2
            pct_below = 100 * (diff < 0).mean()
            print(f'  below {label:9s} ceiling:  {pct_below:.1f}% below, '
                  f'mean gap={-diff.mean():.3f}  t={t:.2f}  p(one-sided)={p_one:.3g}  n={len(sub)}')

In [ ]:
# ── PLOT 4: Model comparison — β reliability across embedding models ───────────
def _norm_compare_entry(entry):
    """Accept either a model tag (str) or a (model_tag, layer) tuple."""
    if isinstance(entry, (tuple, list)):
        return entry[0], entry[1]
    return entry, LAYER

model_stats = []
for entry in COMPARE_MODELS:
    m, layer = _norm_compare_entry(entry)
    df = load_agg(m, layer=layer)
    if df is None or 'fold_beta_corr' not in df.columns:
        print(f'  {m} (L{layer:02d}): no results'); continue
    label = f'{m}\nL{layer:02d}'
    for region in ['hippocampus', 'ACC']:
        for cond in ['self', 'other']:
            sub = df[(df['region'] == region) & (df['condition'] == cond)]
            if sub.empty: continue
            sig = sub[sub['significant']]
            model_stats.append({
                'model': label, 'region': region, 'condition': cond,
                'med_beta_corr': sig['fold_beta_corr'].median() if len(sig) else np.nan,
                'pct_sig': 100 * sub['significant'].mean(),
                'med_r2': sig['r2'].median() if len(sig) else np.nan,
            })

if model_stats:
    ms = pd.DataFrame(model_stats)
    for region in ['hippocampus', 'ACC']:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        any_plotted = False
        for ax_i, cond in enumerate(['self', 'other']):
            ax = axes[ax_i]
            sub = ms[(ms['region'] == region) & (ms['condition'] == cond)]
            if sub.empty:
                ax.set_visible(False); continue
            any_plotted = True
            x = range(len(sub))
            ax.bar(x, sub['pct_sig'], color='#2166ac', alpha=0.7, label='% sig neurons')
            ax2 = ax.twinx()
            ax2.plot(x, sub['med_beta_corr'], 'o-', color='#d6604d',
                     linewidth=2, markersize=7, label='median β reliability')
            ax.set_xticks(list(x))
            ax.set_xticklabels(sub['model'], rotation=20, ha='right', fontsize=9)
            ax.set_ylabel('% significant neurons', color='#2166ac')
            ax2.set_ylabel('Median cross-fold β corr', color='#d6604d')
            ax.set_title(f'{region} / {cond}')
        if not any_plotted:
            plt.close(fig); continue
        plt.suptitle('Model comparison — significance & β reliability', y=1.01)
        plt.tight_layout()
        plt.savefig(f'../figures/02_model_comparison_{region}.pdf', bbox_inches='tight')
        plt.show()
else:
    print('Run other models first via notebook 01 with COMPARE_MODELS')